## Предсказание вероятности клика (CTR) на рекламный баннер с калибровкой модели

Проект реализует полный ML-пайплайн для задачи бинарной классификации: предсказание вероятности клика пользователя по рекламному объявлению (Click-Through Rate). Особый акцент сделан на калибровке предсказанных вероятностей - ключевом требовании для корректной работы рекламного аукциона.

# Структура проекта

## 1. Подготовка среды и загрузка данных

In [ ]:
# pip install --upgrade scikit-learn
# pip install phik

In [ ]:
# импорт библиотек
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import scipy

import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    cross_val_score, 
    cross_validate,
    StratifiedKFold,
    GridSearchCV
)
from sklearn.feature_selection import (
    VarianceThreshold, 
    RFE,
    mutual_info_classif
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler, 
    OneHotEncoder,
    TargetEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    average_precision_score,
    precision_score, 
    recall_score, 
    f1_score,
    brier_score_loss
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.dummy import DummyClassifier

from sklearn.frozen import FrozenEstimator
from time import time
import phik
import joblib

In [ ]:
# pip freeze > requirements.txt

In [ ]:
print("="*60)
print("Версии библиотек")
print("="*60)
print(f"pandas=={pd.__version__}")
print(f"numpy=={np.__version__}")
print(f"scikit-learn=={sklearn.__version__}")
print(f"matplotlib=={matplotlib.__version__}")
print(f"seaborn=={sns.__version__}")
print(f"phik=={phik.__version__}")
print(f"scipy=={scipy.__version__}")
print(f"joblib=={joblib.__version__}")

In [ ]:
# задание параметров фигур графиков
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)
pd.set_option('display.max_colwidth', 300)

In [ ]:
# фиксация RANDOM_SEED
RANDOM_SEED = 42
print(f"Зафиксированное значение зерна ГПСЧ: {RANDOM_SEED}")

In [ ]:
# загрузка датасета по ссылке
df = pd.read_csv('https://code.s3.yandex.net/datasets/ds_s16_ad_click_dataset.csv')

In [ ]:
# объем данных
print(f"Размер датасета \nколичество строк: {df.shape[0]} \nколичество столбцов: {df.shape[1]}")

In [ ]:
# вывод первых 5 строк
print("="*60)
print("Первые 5 строк")
print("="*60)
print(df.head())

In [ ]:
# типы данных
print("="*60)
print("Информация о типах столбцов")
print("="*60)
print(df.dtypes)

In [ ]:
# Проверка успешности загрузки
print("="*60)
if df.shape[0] == 0:
    print("Датасет пуст")
else:
    print("Загрузка данных прошла успешно")

## 2. Исследовательский анализ данных (EDA)

#### 2.1 Базовая информация о датасете

In [ ]:
# вывод базовой информации
print(f"Число наблюдений в датасете: {df.shape[0]}")
print(f"Число признаков: {df.shape[1] - 1}")

In [ ]:
# числовые и категориальные признаки
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

print("="*60)
print(f"Числовые признаки ({len(num_cols)}): {num_cols}")
print(f"Категориальные признаки ({len(cat_cols)}): {cat_cols}")

In [ ]:
print("="*60)
print("Типы данных")
print(df.dtypes)

#### 2.2 Анализ целевой переменной

In [ ]:
# анализ целевой переменной click
click_counts = df['click'].value_counts()
click_percent = df['click'].value_counts(normalize=True) * 100

print("="*60)
print(f"Объем отрицательного класса (0): {click_counts[0]}")
print(f"Доля отрицательного класса в данных: {click_percent[0]:.2f}")
print(f"Объем положительного класса (1): {click_counts[1]}")
print(f"Доля положительного класса в данных: {click_percent[1]:.2f}")
print(f"\nСоотношение классов: {round(click_counts[0]/click_counts[1], 1)} / 1")
print("="*60)

In [ ]:
# построение столбчатой диаграммы
sns.countplot(data=df, x='click')
plt.title("Распределение целевой переменной по классам")
plt.show()

---
Процент класса `0` порядка 83%, класса `1` - порядка 17%.  
**Соотношение классов: 4.8:1** — присутствует умеренный дисбаланс в пользу отрицательного класса. Задача с дисбалансом, поэтому используем `PR-AUC` как основную метрику, нечувствительную к дисбалансу.

#### 2.3 Анализ признаков

Одними из первых кандидатов на удаление являются высококардинальные категориальные признаки, для которых число уникальных значений стремится к числу наблюдений в данных: такие признаки не несут предсказательной силы. Определим их количество

In [ ]:
print("="*60)
print("Число уникальных значений:")
print("="*60)
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
cardinal = df[cat_cols].nunique().sort_values(ascending=False)
cardinal

Категориальные столбцы с высокой кардинальностью, где доля уникальных значений превышает 80%, можно в будущем удалить:

In [ ]:
cols_to_drop = cardinal[(cardinal / max(cardinal)) > .8].index.to_list()
print("="*60)
print(f"Высококардинальные столбцы для удаления ({len(cols_to_drop)}):")
print("="*60)
print(cols_to_drop)

Отметим, что столбец `id`, являющийся уникальным идентификатором каждой строки, не несет предсказательной силы и рекомендован к удалению уже на данном этапе.

In [ ]:
# удаление
df.drop(columns='id', inplace=True)
print(f"Размер после удаления: {df.shape[0]} строк и {df.shape[1] - 1} признаков")

Выведем названия числовых и категориальных признаков с учетом удаления:

In [ ]:
# числовые и категориальные признаки
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols.remove('click')
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

print("="*60)
print(f"Числовые признаки ({len(num_cols)}): {num_cols}")
print("="*60)
print(f"Категориальные признаки ({len(cat_cols)}): {cat_cols}")

---
Таким образом, удалили уникальный идентификатор `id` и рекомендовали столбец `device_ip` с высокой кардинальностью (более 80% значений уникальны) к удалению на будущих этапах проекта.

#### 2.4 Анализ пропущенных значений

In [ ]:
# пропуски в столбцах
missing = df.isna().sum()
print("="*60)
print("Процент пропусков в столбцах")
print("="*60)
print(round(missing / len(df) * 100, 2))

In [ ]:
# выведем столбцы с пропусками
missing[missing > 0].index.to_list()

---
Пропущенных значений нет ни в одном признаке. Данные полностью заполнены - предобработка пропусков не требуется.

#### 2.5 Анализ категориальных признаков

In [ ]:
OHE_boundary = 10

for col in cat_cols:
    encoding = None
    if df[col].nunique() > OHE_boundary:
        print(f"Для столбца {col} метод кодирования - Target Encoding\n")
    else:
        print(f"Для столбца {col} метод кодирования - One-hot Encoding\n")

Для столбцов с высокой кардинальностью (более 10 уникальных значений) применим метод `Target` кодирования во избежание неконтролируемого роста размерности датасета. Для остальных (`ml_feature_2`, `ml_feature_7`) - метод `One-hot` кодирования.

#### 2.6 Анализ выбросов и распределений

In [ ]:
# выведем статистики признаков
print("="*60)
print("Статистика числовых признаков:")
print("="*60)
print(df[num_cols].describe())

Для каждого из столбцов определим:  
- количество выбросов, выходящих за отрезок $[Q_1 - 1.5 * IQR; Q_3 + 1.5 * IQR]$;
- процент выбросов относительно полного объема строк в столбце;
- меру скошенности распределения, рассчитанную по коэффициенту асимметрии Фишера (`skew`).

In [ ]:
outlier_info = {}
for col in num_cols:
    # квартили
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    
    # межквартильный размах
    IQR = Q3 - Q1
    
    # границы
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    # число и процент выбросов
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    pct_out = n_out / len(df) * 100
    
    # мера скошенности
    skew = df[col].skew()
    
    # объединение в словарь
    outlier_info[col] = {
        'n_outliers': int(n_out),
        'pct_outliers': round(pct_out, 2),
        'skew': round(skew, 3)
    }
    print("="*60)
    print(f"В столбце {col} \nколичество выбросов = {n_out} \nпроцент выбросов = {pct_out:.2f}% \nмера скошенности = {skew:.3f}\n")

In [ ]:
# выведем графики распределений числовых признаков
df[num_cols].hist(
    figsize=(18, 15),
    grid=False,
    bins=30
)
plt.tight_layout()
plt.show()

Проанализируем распределения признаков:
- hour: Отметим цикличность пиков. Характерно для периодического повторения показа баннеров
- группа анонимизированных признаков C1, C15, C16: большинство наблюдений сконцентрировано вокруг одного значения
- device_type: доминирует категория 1, остальные значения практически не встречаются на общем фоне
- banner_pos: дискретное множество представлено в основном значениями 0 и 1; остальные значения - малочисленны
- device_conn_type: ситуация схожа с banner_pos
- C18: Четыре дискретных значения (0, 1, 2, 3). Наибольшая частота у 0

Отдельно выделим признаки с более непрерывным распределением:
- ml_feature 1, 5, 9, 10: имеют классическое нормальное распределение
- ml_feature 3, 6, 8: близки к константным
- ml_feature_4: бинарный флаг
- C14, C17, C21: Распределения имеют сложную форму с локальными максимумами сразу на нескольких значениях. Судя по параметру `skew` признаки C14, C17 имеют заметный хвост слева, C21 - справа
- C19: Распределение, похожее на пуассоновское с сильным правым хвостом
- C20: ярко выраженное бимодальное распределение, но поскольку один из пиков в районе значения 100000, не похоже на бинарный флаг.

---
При асимметричных распределениях с длинными хвостами и нестандартным мульттмодальным характером выбросы скорее являются фичей данных, чем багом.  
Данные имеют разный масштаб и требуют стандартизации.

#### 2.7 Корреляции

Вычисление матрицы phi_k при наличии категориальных признаков высокой кардинальности - трудоемкая задача. Пожертвуем полнотой анализа и определим матрицу корреляций для всех числовых столбцов, включая таргет, а также для категориальных с уровнем кардинальности <= 1500.

In [ ]:
interval_cols = [
    'click',
    'hour',
    'C1',
    'banner_pos',
    'device_type',
    'device_conn_type',
    'C14', 'C15', 'C16',
    'C17', 'C18', 'C19',
    'C20', 'C21',
    'ml_feature_1',
    'ml_feature_3',
    'ml_feature_4',
    'ml_feature_5',
    'ml_feature_6',
    'ml_feature_8',
    'ml_feature_9',
    'ml_feature_10'
]

In [ ]:
# выборка столбцов для анализа
num_phik_cols = num_cols
cat_phik_cols = [col for col in cat_cols if df[col].nunique() <= 1500]
phik_cols = num_phik_cols + cat_phik_cols + ['click']

In [ ]:
# вычисление матрицы phi_k
corr_matrix = df[phik_cols].phik_matrix(
    interval_cols=num_phik_cols+['click']
)

In [ ]:
plt.figure(figsize=(16, 12))
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f', cmap='YlOrRd', vmin=0, vmax=1, annot_kws={'size': 7}
)
plt.title('Корреляция между признаками и целевой переменной')
plt.show()

In [ ]:
print("="*60)
print("Корреляции признаков с целевой переменной")
print("="*60)
round(corr_matrix['click'][corr_matrix.index != 'click'].sort_values(ascending=False), 2)

Определим сильно скоррелированные пары признаков.

In [ ]:
corr_threshold = .9
phik_cols = corr_matrix.columns
high_corr_pairs = []

for i in range(len(phik_cols)):
    for j in range(i+1, len(phik_cols)):
        if corr_matrix.loc[phik_cols[i], phik_cols[j]] > corr_threshold:
            high_corr_pairs.append((phik_cols[i], phik_cols[j]))
            
print("="*60)
print(f"Пары сильно скоррелированных признаков, где phi_k > 0.9 ({len(high_corr_pairs)} шт.)")
print("="*60)
high_corr_pairs

---
Признаки `ml_feature_` не имеют выраженной связи с остальными и между собой, но для некоторых из них коэффициент `phi_k` ненулевой (в районе 0.1-0.2).  
Признаки `C18`, `C21`, `site_id`, `site_domain`, `app_id` на фоне остальных показывают заметную корреляцию с таргетом (коэффициент от 0.2 до 0.34)  
Выявлено 14 пар сильно скоррелированных признаков, среди которых оставлять в будущем лучше тот, что сильнее скоррелирован с таргетом.

#### 2.8 Выводы по EDA

1. Дисбаланс классов умеренный (4.8:1). PR-AUC - основная метрика.
2. Удален 1 признак без предсказательной способности: уникальный идентификатор `id`.
3. Пропусков нет - предобработка пропусков не требуется.
4. Наиболее информативные признаки по phi_k с `click`: `site_id`, `site_domain`, `C18`, `app_id`, `ml_feature_9`, `ml_feature_10`, `C21`, `app_domain`, `app_category`, `site_category`, `ml_feature_8`, `device_conn_type`, `C16`, `C14`, `C15`.
5. 14 пар сильно скоррелированных признаков - кандидаты на удаление одного из каждой пары.
6. Высококардинальные категориальные признаки `device_id`, `device_model`, `site_id`, `site_domain`, `app_id`, `app_domain`, `app_category`, `site_category` требуют Target Encoding, остальные - OHE.


**Необходимые действия по предобработке**

1. Масштабирование числовых признаков для линейных моделей; выберем стандартизацию.
2. Target Encoding для высококардинальных категориальных признаков.
3. One-Hot Encoding для `ml_feature_2`, `ml_feature_7`.
4. Рассмотреть удаление одного признака из каждой высококоррелированной пары на этапе отбора признаков.

## 3. Разделение данных на выборки

In [ ]:
# разделение на признаки и целевую переменную
X = df.drop(columns=['click'])
y = df['click']

In [ ]:
# стратифицированное разделение
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y
)

In [ ]:
# выведем размеры выборок
print(f"Размер обучающей выборки: {X_train.shape[0]} строк")
print(f"Размер тестовой выборки: {X_test.shape[0]} строк")

# доля положительного класса
print(f"Процент положительного класса в обучающей выборке: {y_train.mean()*100:.2f}")
print(f"Процент положительного класса в тестовой выборке: {y_test.mean()*100:.2f}")

Баланс классов сохранен посредством стратификации.

---

## 4. Предобработка данных — построение пайплайнов

#### 4.1 Создайние пайплайна для предобработки данных

In [ ]:
# бинарные фичи
binary_features = [
    'ml_feature_4'
]

# численные непрерывные фичи
numeric_features = num_cols
numeric_features.remove('ml_feature_4')

# категориальные фичи для OneHot кодирования
onehot_cat_features = [col for col in cat_cols if df[col].nunique() <= 10]

# категориальные фичи для Target кодирования
target_cat_features = [col for col in cat_cols if col not in onehot_cat_features]

print("="*60)
print(f"Бинарные фичи ({len(binary_features)} шт.):")
print(binary_features)
print("="*60)
print(f"Численные непрерывные фичи ({len(numeric_features)} шт.):")
print(numeric_features)
print("="*60)
print(f"One-hot фичи ({len(onehot_cat_features)} шт.):")
print(onehot_cat_features)
print("="*60)
print(f"Target фичи ({len(target_cat_features)} шт.):")
print(target_cat_features)
print("="*60)

#### 4.2 Объединение пайплайнов

Поскольку в признаках нет пропусков, заполнение проводить не стоит. Но обученная модель пойдет в продакшн, и по мере ее эксплуатации могут поступить данные с пропусками. Для предупреждения такой ситуации добавим `SimpleImputer` со стратегией заполнения модой для бинарных и категориальных признаков и стратегией заполнения медианой для числовых признаков.

In [ ]:
# pipeline бинарных признаков
binary_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

# pipeline численных признаков
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# pipeline категориальных признаков
onehot_cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot_encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# pipeline для категориального признака с большим количеством уник. значений
target_cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('target_encoder', TargetEncoder(
        cv=5,
        smooth=30.0,
        random_state=RANDOM_SEED
    ))
])

In [ ]:
# объединение пайплайов
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('bin', binary_pipeline, binary_features),
        ('onehot', onehot_cat_pipeline, onehot_cat_features),
        ('target', target_cat_pipeline, target_cat_features)
    ],
    remainder='drop'
).set_output(transform='pandas')

## 5. Отбор признаков

Отбор признаков ведем на основании статистик, вычисленных на **тренировочной выборке**.

#### 5.1 Применение фильтрационных методов

Оценку корреляций с последующими выводами об удалении признака следует проводить только на тренировочной выборке. Одна из стратегий - вычисление phik_matrix на всей "сырой" тренировочной выборке. В случае датасета средних размеров при наличии высококардинальных признаков это слишком трудоемкая задача.  
Перед оценкой корреляций воспользуемся **TargetEncoder с кросс-валидацией** для предотвращения случая, когда таргет-оценка для самой многочисленной категории просто практически повторит ее глобальное среднее и коэффициент phik, ровно как и MI, улетят "в космос".  
При этом остальные признаки оставим без изменений.

**Target encoding с кросс-валидацией**

Повторно выведем высококардинальные признаки:

In [ ]:
cardinal

In [ ]:
high_cardinality_features = cardinal[cardinal>10].index.to_list()
high_cardinality_features

In [ ]:
# энкодер с заполнением пропусков
target_cv_encoder = ColumnTransformer(
    transformers=[
        ('target_cv', target_cat_pipeline, high_cardinality_features)
    ],
    remainder='passthrough'
).set_output(transform='pandas')

In [ ]:
X_train_encoded = target_cv_encoder.fit_transform(X_train, y_train)

**Корреляции**

Определим коэффициенты корреляции признаков обработанной тренировочной выборки с таргетом

In [ ]:
# объединяем признаки и таргет
df_phik = X_train_encoded.copy()
df_phik["click"] = y_train

In [ ]:
corr_matrix_processed = df_phik.phik_matrix()

In [ ]:
# выводим корреляции с таргетом
corr_click = corr_matrix_processed['click'][corr_matrix_processed['click'].index != 'click'].sort_values(ascending=False)

# коэффициенты phi_k
corr_click

In [ ]:
phik_to_drop = corr_click[corr_click == 0.0].index.to_list()
print("="*60)
print(f"Признаки с нулевым коэффициентом phi_k ({len(phik_to_drop)} шт.)")
phik_to_drop

**Константные признаки**

Применение класса VarianceThreshold лишено смысла, если признаки масштабированы (особенно, если их стандартизировали и привели к единичной дисперсии). Применяем VarianceThreshold к немасштабированной закодированной тренировочной выборке.

Определим, какие столбцы в закодированной выборке являются числовыми, какие - категориальными.

In [ ]:
num_encoded_cols = X_train_encoded.select_dtypes(include=[np.number]).columns.tolist()
cat_encoded_cols = X_train_encoded.select_dtypes(include='object').columns.tolist()

In [ ]:
# выявление константных числовых признаков
vt = VarianceThreshold(threshold=.0)
vt.fit(X_train_encoded[num_encoded_cols])
num_const_to_drop = [col for col, keep in zip(X_train_encoded[num_encoded_cols].columns, vt.get_support()) if not keep]

print("="*60)
print(f"Константные числовые признаки ({len(num_const_to_drop)} шт.)")
num_const_to_drop

In [ ]:
# выявление константных категориальных признаков
cat_const_to_drop = [col for col in X_train_encoded.select_dtypes(include='object').columns if X_train_encoded[col].nunique() == 1]

print("="*60)
print(f"Константные категориальные признаки ({len(cat_const_to_drop)} шт.)")
cat_const_to_drop

**Квазиконстантные признаки**

In [ ]:
# выявление квазиконстантных числовых признаков
vt_quasi = VarianceThreshold(threshold=.001)
vt_quasi.fit(X_train_encoded[num_encoded_cols])
num_quasiconst_to_drop = [col for col, keep in zip(X_train_encoded[num_encoded_cols].columns, vt_quasi.get_support()) if not keep]

print("="*60)
print(f"Квазиконстантные числовые признаки ({len(num_quasiconst_to_drop)} шт.)")
num_quasiconst_to_drop

In [ ]:
# выявление константных категориальных признаков
cat_quasiconst_to_drop = [col for col in X_train_encoded.select_dtypes(include='object').columns if (X_train_encoded[col].nunique() / len(X_train_encoded[col])) > .9]

print("="*60)
print(f"Квазиконстантные категориальные признаки ({len(cat_quasiconst_to_drop)} шт.)")
cat_quasiconst_to_drop

---
Отметим **особенность** работы TargetEncoder с кросс-валидацией.
- при кодировании высококардинального `device_ip` с кросс валидацией значение таргета для текущей (и почти всегда уникальной в случае `device_ip`) категории рассчитывалась по остальным фолдам, т.е. по фолдам, где нет текущей категории. Из-за этого почти для всех значений (а они почти все уникальны) было подставлено глобальное среднее таргета, которое практически повторяет долю положительного класса. Получили квази-константу. Для малой доли неуникальных значений, встречающихся в `device_ip`, кодирование могло дать что угодно, в зависимости от того, какой таргет был в рамках этой категории. Получили в итоге ситуацию где для всех уникальных категорий энкодер дал среднее таргета, а для остальных категорий - кучу выбросов. Под эти выбросы будет подстраиваться любая линейная модель и переучиваться (в том числе в грядущем RFE, где этот признак получит скорее всего хороший ранг и останется после отбора). 
- при кодировании `device_ip` мы получили по сути ту же картину, но по диаметрально противоположной причине: подавляющее большинство значений были уже одинаковы в сырых данных. При их кодировании волей-неволей получится практически глобальное среднее таргета, а для остальных значений - что угодно.

Эти признаки надо удалить из данных уже сейчас, потому что на этапе RFE мы уже получим переобучение и аномально низкий ranking для `device_ip` и `device_ip` (т.е. метод точно отберет эти признаки по важности).

In [ ]:
target_cat_features.remove('device_id')
target_cat_features.remove('device_ip')

#### 5.2 Применение методов-обёрток

Поскольку estimator'ом в методе-обертке будет выбрана линейная модель, необходимо передать ему числовые масштабированные признаки. Применим к тренировочной выборке препроцессор, написанный в пункте 4.

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train, y_train)

В качестве метода-обертки выберем RFE, поскольку он является балансом между прямым и обратным методами.

In [ ]:
num_features_to_select = 20

model = LogisticRegression(max_iter=1000, solver='liblinear', class_weight='balanced', random_state=RANDOM_SEED)

rfe_selector = RFE(estimator=model,
                   n_features_to_select=num_features_to_select,
                   step=1,
                   verbose=0)

# обучаем RFE на обучающих данных
rfe_selector.fit(X_train_processed, y_train)


In [ ]:
# Получаем информацию о выбранных признаках
selected_rfe_features = X_train_processed.columns[rfe_selector.support_].tolist()
print(f'\nВыбранные признаки RFE ({num_features_to_select}): {selected_rfe_features}')
print("="*60)
print()
for feature, rank in zip(X_train_processed.columns, rfe_selector.ranking_):
    print(f'Признак: {feature}, Ранг: {rank}')

#### 5.3 Выбор финального набора признаков

Результат анализа корреляций:

In [ ]:
phik_to_drop = corr_click[corr_click == 0.0].index.to_list()
print("="*60)
print(f"Признаки с нулевым коэффициентом phi_k ({len(phik_to_drop)} шт.)")
phik_to_drop

Константных признаков нет. Квазиконстантные признаки:

In [ ]:
print("="*60)
print(f"Квазиконстантные числовые признаки ({len(num_quasiconst_to_drop)} шт.)")
num_quasiconst_to_drop

Результат RFE трактовать затруднительно. Отобранные признаки - лишь результат обучения конкретной модели с конкретными параметрами и при конкретных настройках оберточного метода. Можем лишь удостовериться в том, что признаки с нулевой корреляцией phi_k (`ml_feature_4`, `ml_feature_3`, `ml_feature_1`) в совокупности и с другими признаками (то есть внутри конкретной ML-модели) получили небольшую значимости (ranking RFE равны 17, 16 и 13 соответственно)

---

Также для упрощения модели и борьбы с мультиколлинеарностью принято решение помимо признаков с нулевой корреляцией (`ml_feature_4`, `ml_feature_3`, `ml_feature_1`) и квазиконтсантных после кодирования (`device_ip`, `device_id`) удалить пары сильно скоррелированных признаков.  
Логика удаления была следующей: определялись группы признаков, связанных сильной корреляцией phi_k между собой. Среди них оставляем те, что сильнее других в своей группе связаны с таргетом.  
Таким образом было дополнительно удалено: `banner_pos`, `C1`, `device_type`, `C17`, `site_domain`, `site_category`, `app_domain`, `app_category`. 

In [ ]:
# для удаления достаточно не включать признаки в вывод препроцессора
numeric_features = [col for col in numeric_features if col not in ['ml_feature_3', 'ml_feature_1', 'banner_pos', 'C1', 'device_type', 'C17']]
binary_features = [col for col in binary_features if col not in ['ml_feature_4']]
target_cat_features = [col for col in target_cat_features if col not in ['site_domain', 'site_category', 'app_domain', 'app_category']]
print(numeric_features)
print(binary_features)
print(onehot_cat_features)
print(target_cat_features)

In [ ]:
# переобозначаем препроцессор
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('bin', binary_pipeline, binary_features),
        ('onehot', onehot_cat_pipeline, onehot_cat_features),
        ('target', target_cat_pipeline, target_cat_features)
    ],
    remainder='drop'
).set_output(transform='pandas')

---
На этапе отбора признаков удалены:
- числовые `ml_feature_4`, `ml_feature_3`, `ml_feature_1` (нулевая линейная связь с таргетом и малый вес по выводам модели-обертки
- категориальные `device_ip`, `device_id` (высокая кардинальность и переобучение после TargetEncoder)
- сильно скоррелированные `banner_pos`, `C1`, `device_type`, `C17`, `site_domain`, `site_category`, `app_domain`, `app_category`

In [ ]:
print("="*60)
print(f"Дальнейшее обучение модели происходит на {len(numeric_features) + len(binary_features) + len(onehot_cat_features) + len(target_cat_features)} признаках")
print("="*60)

## 6. Обучение базовой модели

In [ ]:
# стратегия разбиения на фолды со стратификацией 
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# метрики, замеряемые на тестовых фолдах
scoring = {
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'pr_auc': 'average_precision'
}

#### 6.1 Обучение `DummyClassifier`

In [ ]:
# pipeline препроцессор + модель
dummy_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DummyClassifier(strategy='stratified', random_state=RANDOM_SEED))
])

# кросс-валидация
dummy_res = cross_validate(
    estimator=dummy_pipeline,
    X=X_train, y=y_train,
    cv=cv,
    n_jobs=-1,
    scoring=scoring
)

print("="*60)
print(f"PR-AUC по фолдам: {dummy_res['test_pr_auc'].round(3)}")

print("="*60)
print(f"PR-AUC mean +- std: {dummy_res['test_pr_auc'].mean():.4f} +- {dummy_res['test_pr_auc'].std():.4f}")

#### 6.2 Обучение `LogisticRegression`

In [ ]:
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        max_iter=1000,
        solver='liblinear',
        class_weight='balanced',
        random_state=RANDOM_SEED
    ))
])

# кросс-валидация
lr_res = cross_validate(
    estimator=lr_pipeline,
    X=X_train, y=y_train,
    cv=cv,
    n_jobs=-1,
    scoring=scoring
)

print("="*60)
print(f"PR-AUC по фолдам: {lr_res['test_pr_auc'].round(3)}")

print("="*60)
print(f"PR-AUC mean +- std: {lr_res['test_pr_auc'].mean():.3f} +- {lr_res['test_pr_auc'].std():.3f}")
print(f"Precision: {lr_res['test_precision'].mean():.3f}")
print(f"Recall: {lr_res['test_recall'].mean():.3f}")
print(f"F1: {lr_res['test_f1'].mean():.3f}")

#### 6.3 Обучение `SVC`

In [ ]:
svc_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LinearSVC(
        max_iter=5000,
        tol=1e-3,
        dual=False,
        class_weight='balanced',
        random_state=RANDOM_SEED
    ))
])

# кросс-валидация
svc_res = cross_validate(
    estimator=svc_pipeline,
    X=X_train, y=y_train,
    cv=cv,
    n_jobs=-1,
    scoring=scoring
)

print("="*60)
print(f"PR-AUC по фолдам: {svc_res['test_pr_auc'].round(3)}")

print("="*60)
print(f"PR-AUC mean +- std: {svc_res['test_pr_auc'].mean():.3f} +- {svc_res['test_pr_auc'].std():.3f}")
print(f"Precision: {svc_res['test_precision'].mean():.3f}")
print(f"Recall: {svc_res['test_recall'].mean():.3f}")
print(f"F1: {svc_res['test_f1'].mean():.3f}")

#### 6.4 Сравнение моделей

In [ ]:
compare_models = pd.DataFrame({
    'Модель': ['DummyClassifier', 'LogisticRegression', 'LinearSVC'],
    'PR-AUC': [round(dummy_res['test_pr_auc'].mean(), 4), round(lr_res['test_pr_auc'].mean(), 4), round(svc_res['test_pr_auc'].mean(), 4)]
})

print("="*60)
print("Сравнение моделей по PR-AUC")
compare_models

---
- LogisticRegression лучше DummyClassifier: PR-AUC вырос с 0.171 до 0.414, то есть в 2.42 раза. Модель логрега улучшила способность предсказывать клики относительно случайного угадывания.
- LinearSVC практически неотличим от LogisticRegression: разница в PR-AUC порядка 0.0004. Обе модели линейные и работают с одинаковым набором признаков, поэтому такой результат ожидаем.

- Значение PR-AUC 0.41 - умеренный результат для задачи предсказания кликов с таким набором признаков. Линейные модели в 2.4 раза лучше baseline и очевидно уловили закономерности.

- Поскольку признаки анонимизированы, задачу feature_engineering не реализовать. Одним из способов увеличения качества модели может стать удаление сильно скоррелированных признаков и применение нелинейных ядер в SVM.

## 7. Подбор гиперпараметров: Grid Search с кросс-валидацией

#### 7.1 Определение сетки гиперпараметров

In [ ]:
# сетка для LogReg
lr_param_grid = {
    'lr__C': [0.1, 1.0],
    'lr__penalty': ['l1', 'l2'],
    'lr__class_weight': ['balanced', None],
    'lr__solver': ['liblinear'],
    'lr__max_iter': [100, 1000]
}

In [ ]:
# сетка для LinearSVC
svc_param_grid = {
    'svc__C': [0.001, 0.01, 0.1, 1],
    'svc__class_weight': [None, 'balanced'],
    'svc__dual': ['auto', False],
    'svc__max_iter': [1000, 5000]
}

In [ ]:
# сетка для SVC
svc_hard_param_grid = {
    'svc_hard__C': [0.1, 1, 10],
    'svc_hard__kernel': ['rbf', 'poly'],
    'svc_hard__gamma': ['scale', 'auto']
}

In [ ]:
lr_comb = 5 * 2 * 2 * 2
svc_comb = 3 * 5 * 2
print(f"LogisticRegression: {lr_comb} комбинаций")
print(f"SVC: {svc_comb} комбинаций")

#### 7.2 Применение Grid Search

**Варьирование параметров LogisticRegression**

In [ ]:
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('lr', LogisticRegression(random_state=RANDOM_SEED))
])

lr_grid_search = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=lr_param_grid,
    scoring='average_precision',
    cv=cv,
    n_jobs=-1
)

In [ ]:
t1 = time()
lr_grid_search.fit(X_train, y_train)
t2 = time()
print(f"Лучшие параметры LR:")
for param, value in lr_grid_search.best_params_.items():
    print(f"{param}: {value}")
print(f"Лучший PR-AUC: {lr_grid_search.best_score_:.4f}")
print(f"Готово за {round((t2 - t1) // 60)} минут {round((t2 - t1) % 60)} секунд")

**Варьирование параметров LinearSVC**

In [ ]:
svc_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('svc', LinearSVC(random_state=RANDOM_SEED))
])

svc_grid_search = GridSearchCV(
    estimator=svc_pipeline,
    param_grid=svc_param_grid,
    scoring='average_precision',
    cv=cv,
    n_jobs=-1
)

In [ ]:
t1 = time()
svc_grid_search.fit(X_train, y_train)
t2 = time()

print(f"Лучшие параметры SVC:")
for param, value in svc_grid_search.best_params_.items():
    print(f"{param}: {value}")
print(f"Лучший PR-AUC: {svc_grid_search.best_score_:.4f}")
print(f"Готово за {round((t2 - t1) // 60)} минут {round((t2 - t1) % 60)} секунд")

**Варьирование параметров SVC с нелинейными ядрами**

In [ ]:
svc_hard_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('svc_hard', SVC(random_state=RANDOM_SEED))
])

svc_hard_grid_search = GridSearchCV(
    estimator=svc_hard_pipeline,
    param_grid=svc_hard_param_grid,
    scoring='average_precision',
    cv=cv,
    n_jobs=-1
)

Для экономии ресурса вычленим малую выборку из тренировочной:

In [ ]:
# стратифицированное разделение
X_train_svc, X_temp, y_train_svc, y_temp = train_test_split(
    X_train, y_train,
    test_size=0.75,
    random_state=RANDOM_SEED,
    stratify=y_train
)

In [ ]:
t1 = time()
svc_hard_grid_search.fit(X_train_svc, y_train_svc)
t2 = time()

print(f"Лучшие параметры SVC:")
for param, value in svc_hard_grid_search.best_params_.items():
    print(f"{param}: {value}")
print(f"Лучший PR-AUC: {svc_hard_grid_search.best_score_:.4f}")
print(f"Готово за {round((t2 - t1) // 60)} минут {round((t2 - t1) % 60)} секунд")

---
Метрики тяжелой модели SVC с нелинейным ядром критически хуже метрик линейных моделей. Далее SVC с rbf не разбираем.

#### 7.3 Составление таблицы результатов

Лучшие результаты LogReg

In [ ]:
# DataFrame для удобства обработки
lr_df = pd.DataFrame(lr_grid_search.cv_results_)

In [ ]:
top_10_lr = lr_df[[
    'params', 
    'mean_test_score', 
    'std_test_score', 
    'rank_test_score'
]].sort_values(by='rank_test_score').head(10).reset_index(drop=True)
print("="*60)
print("Топ-10 наборов параметров для LogisticRegression:")
display(top_10_lr)

Лучшие результаты LinearSVC

In [ ]:
# DataFrame для удобства обработки
svc_df = pd.DataFrame(svc_grid_search.cv_results_)

In [ ]:
top_10_svc = svc_df[[
    'params', 
    'mean_test_score', 
    'std_test_score', 
    'rank_test_score'
]].sort_values(by='rank_test_score').head(10).reset_index(drop=True)
print("="*60)
print("Топ-10 наборов параметров для LinearSVC:")
display(top_10_svc)

## 8. Финальная модель

#### 8.1 Обучение финальной модели

К лучшей модели+препроцессору обратимся через атрибут best_estimator_ класса GridSearch:

**LogisticRegression**

In [ ]:
# лучшая модель с препроцессором
lr_best = lr_grid_search.best_estimator_

lr_best.fit(X_train, y_train)
y_lr_score = lr_best.predict_proba(X_test)[:, 1]

**LinearSVC**

Поскольку в рамках структуры проекта изначально не требовалось возвращать скоры, воспользуемся decision_function. Она подойдет для расчета PR-AUC, так как сохраняет ранжирование, хоть и возвращает расстояния.

In [ ]:
# лучшая модель с препроцессором
svc_best = svc_grid_search.best_estimator_

svc_best.fit(X_train, y_train)
y_svc_score = svc_best.decision_function(X_test)

#### 8.2 Расчет метрик на тестовой выборке

**LogisticRegression**

In [ ]:
PR_AUC_lr = average_precision_score(y_test, y_lr_score)
brier_score_lr = brier_score_loss(y_test, y_lr_score)

**LinearSVC**

Дя расчета оценки Бриера требуются вероятности. Применим сигмоидальное преобразование для заключения результатов decision_function в интервал (0; 1).

In [ ]:
# расчет сырых вероятностей через сигмоиду
y_svc_score_sigm = 1 / (1 + np.e**(-y_svc_score))

In [ ]:
PR_AUC_svc = average_precision_score(y_test, y_svc_score)
brier_score_svc = brier_score_loss(y_test, y_svc_score_sigm)

Сравнение

In [ ]:
print("="*60)
print(f"PR-AUC логистической регрессии: {round(PR_AUC_lr, 4)}")
print(f"Оценка Бриера логистической регрессии: {round(brier_score_lr, 4)}")
print("="*60)
print(f"PR-AUC LinearSVC: {round(PR_AUC_svc, 4)}")
print(f"Оценка Бриера LinearSVC: {round(brier_score_svc, 4)}")
print("="*60)

---
Основная метрика PR-AUC, учитывающая дисбаланс классов, для обеих моделей на тестовой выборке снизилась, но незначительно. Переобучения нет и модели ведут себя стабильно.  
Оценка Бриера для логистической регрессии ниже, что говорит о более хорошей откалиброванности модели. Предсказанные ею вероятности лучше совпадают с реальной частотой кликов, чем в случае LinearSVC.

#### 8.3 Анализ весов модели

In [ ]:
# df весов логрега
lr_coef = pd.DataFrame({
    'feature': lr_best['lr'].feature_names_in_,
    'coefficient_lr': lr_best['lr'].coef_[0]
})

In [ ]:
# df весов LinearSVC
svc_coef = pd.DataFrame({
    'feature': svc_best['svc'].feature_names_in_,
    'coefficient_svc': svc_best['svc'].coef_[0]
})

Сравним коэффициенты, вычисленные для каждого из признаков разными моделями классификации:

In [ ]:
coef_comparison = pd.merge(lr_coef, svc_coef, on='feature', how = 'inner').sort_values(by='coefficient_lr', ascending=False)
print("="*60)
print("Сравнение коэффициентов")
print("="*60)
coef_comparison 

---
Заметим, что самые важные признаки для моделей одинаковы что в случае LogisticRegression, что в случае LinearSVC. Топ 5 признаков по важности для обеих моделей:
- `app_id`, `site_id`, `device_model`, `ml_feature_9`, `ml_feature_10`  

Поскольку три из топ-5 признаков получены кодированием через таргет, а два машинно сгенерированы, интерпретировать связь признака и таргета через веса проблематично. 

## 9. Калибровка модели

#### 9.1 Проверка текущей калибровки

**Проверка калибровки LogisticRegression**

In [ ]:
# вычисление точек диаграммы
prob_lr_true, prob_lr_pred = calibration_curve(
    y_test,
    y_lr_score,
    n_bins=10
)
# диаграмма калибровки для LogReg
sns.lineplot(x=prob_lr_pred, y=prob_lr_true, marker='o', label='LogReg')
# диаграмма идеально откалиброванной модели
sns.lineplot(x=[0, 1], y=[0, 1], linestyle='--', label='Идеальная калибровка')
plt.xlabel("Предсказанная вероятность")
plt.ylabel("Фактическая вероятность")
plt.grid()
plt.show()

**Проверка калибровки LinearSVC**

In [ ]:
# вычисление точек диаграммы
prob_svc_true, prob_svc_pred = calibration_curve(
    y_test,
    y_svc_score_sigm,
    n_bins=10
)
# диаграмма калибровки для LinearSVC
sns.lineplot(x=prob_svc_pred, y=prob_svc_true, marker='o', label='LinearSVC')
# диаграмма идеально откалиброванной модели
sns.lineplot(x=[0, 1], y=[0, 1], linestyle='--', label='Идеальная калибровка')
plt.xlabel("Предсказанная вероятность")
plt.ylabel("Фактическая вероятность")
plt.grid()
plt.show()

#### 9.2 Применение методов калибровки

Откалибруем модель LinearSVC как модель с высокой оценкой Бриера (плохой калибровкой по умолчанию). Для процедуры по заданию необходимо использовать отдельную калибровочную выборку. Выделим ее:

In [ ]:
X_train_small, X_calib, y_train_small, y_calib = train_test_split(
    X_train, y_train,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=y_train
)

In [ ]:
svc_best.fit(X_train_small, y_train_small)
frozen = FrozenEstimator(svc_best)
# передаем предобученный эстиматор и запрещаем трогать его веса
calib_svc = CalibratedClassifierCV(estimator=frozen, method='isotonic', cv=cv)

In [ ]:
calib_svc.fit(X_calib, y_calib)
# получение откалиброванных вероятностей
y_svc_score_calib = calib_svc.predict_proba(X_test)[:, 1]

In [ ]:
# вычисление точек диаграммы
prob_svc_calib_true, prob_svc_calib_pred = calibration_curve(
    y_test,
    y_svc_score_calib,
    n_bins=10
)
# диаграмма калибровки для откалиброванной LinearSVC
sns.lineplot(x=prob_svc_calib_pred, y=prob_svc_calib_true, marker='o', label='LinearSVC_откалиброванная')
# диаграмма идеально откалиброванной модели
sns.lineplot(x=[0, 1], y=[0, 1], linestyle='--', label='Идеальная калибровка')
# диаграмма калибровки для LinearSVC
sns.lineplot(x=prob_svc_pred, y=prob_svc_true, marker='o', label='LinearSVC')
plt.xlabel("Предсказанная вероятность")
plt.ylabel("Фактическая вероятность")
plt.grid()
plt.show()

---
Наглядно видно, как сработал калибратор с кросс-валидацией. Откалиброванные вероятности устремились к идеальной диаграмме, пропала широкая область, где модель переоценивала вероятности.

#### 9.3 Сравнение моделей до и после калибровки

In [ ]:
print("="*60)
print(f"Оценка Бриера для LinearSVC до калибровки {round(brier_score_loss(y_test, y_svc_score_sigm), 3)}")
print("="*60)
print(f"Оценка Бриера для LinearSVC после калибровки {round(brier_score_loss(y_test, y_svc_score_calib), 3)}")
print("="*60)

---
Оценка Бриера для LinearSVC после калибровки сравнялась с оценкой для хорошо откалиброванной модели LogisticRegression.

## 10. Оценка качества калибровки

#### 10.1 Расчет метрик калибровки

Реализуем вспомогательную функцию расчета ECE и MCE

In [ ]:
def calculate_ece_mce(y_true, y_prob, n_bins=10):
    """
    Рассчитывает ECE, MCE
    """
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0
    max_error = 0
    n = len(y_true)
    for i, (bin_lower, bin_upper) in enumerate(zip(bins[:-1], bins[1:])):
        if i == n_bins - 1:
            mask = (y_prob >= bin_lower) & (y_prob <= bin_upper)
        else:
            mask = (y_prob >= bin_lower) & (y_prob < bin_upper)
        if np.sum(mask) > 0:
            bin_conf = np.mean(y_prob[mask])
            bin_acc = np.mean(y_true[mask])
            ece += np.sum(mask) * np.abs(bin_conf - bin_acc)
            max_error = max(max_error, np.abs(bin_conf - bin_acc))
    return ece / n, max_error

#### 10.2 Сравнение моделей до и после калибровки

In [ ]:
calib_metrics = pd.DataFrame({
    'Model': ['uncalibrated_LinearSVC', 'calibrated_LinearSVC', 'LogisticRegression'],
    'PR_AUC': [average_precision_score(y_test, y_svc_score_sigm), average_precision_score(y_test, y_svc_score_calib), average_precision_score(y_test, y_lr_score)],
    'Brier score': [brier_score_loss(y_test, y_svc_score_sigm), brier_score_loss(y_test, y_svc_score_calib), brier_score_loss(y_test, y_lr_score)],
    'ECE': [calculate_ece_mce(y_test, y_svc_score_sigm)[0], calculate_ece_mce(y_test, y_svc_score_calib)[0], calculate_ece_mce(y_test, y_lr_score)[0]],
    'MCE': [calculate_ece_mce(y_test, y_svc_score_sigm)[1], calculate_ece_mce(y_test, y_svc_score_calib)[1], calculate_ece_mce(y_test, y_lr_score)[1]]
})
calib_metrics[['PR_AUC', 'Brier score', 'ECE', 'MCE']] = calib_metrics[['PR_AUC', 'Brier score', 'ECE', 'MCE']].round(3)

print("="*60)
print("Сравнение моделей до и после калибровки")
print(calib_metrics)
print("="*60)

---
Результат калибровки LinearSVC:
- Значение оценки Бриера после калибровки снизилось с 0.215 до 0.125, что говорит об улучшении качества вероятностей модели в целом.  
- Критически упало среднее расхождение вероятностей ECE, что опять же положительно характеризует качество прогнозируемых моделью вероятностей после калибровки.  
- Максимальная ошикба MCE снизилась незначительно: с 33.2% до 26.7%, так как осталась зона завышения вероятности относительно фактической. 


## 11. Финальный отчёт и выводы

#### 11.1 Сводная таблица результатов

In [ ]:
# отдельно обучим Dummy не через кросс-валидацию, а напрямую
dummy_pipeline.fit(X_train, y_train)
y_dummy = dummy_pipeline.predict(X_test)

In [ ]:
calib_metrics.loc[3] = {
    'Model': 'DummyClassifier',
    'PR_AUC': average_precision_score(y_test, y_dummy),
    'Brier score': brier_score_loss(y_test, y_dummy),
    'ECE': calculate_ece_mce(y_test, y_dummy)[0],
    'MCE': calculate_ece_mce(y_test, y_dummy)[0]
}
calib_metrics[['PR_AUC', 'Brier score', 'ECE', 'MCE']] = calib_metrics[['PR_AUC', 'Brier score', 'ECE', 'MCE']].round(3)

In [ ]:
print("="*60)
print("Сравнение всех моделей")
print("="*60)
calib_metrics

По результатам обоих классификаторов (LinearSVC, LogReg) получены следующие топ-5 важных признаков:

In [ ]:
top_features = [item.replace('target__', '') for item in coef_comparison.head(5)['feature'].to_list()]
top_features = [item.replace('num__', '') for item in top_features]
print("="*60)
print("Самые важные по весам признаки")
print("="*60)
top_features

#### 11.2 Общие выводы

1) Качество моделей на тестовой выборке существенно улучшилось по сравнению с базовой моделью DummyClassifier. Метрика PR-AUC выросла с 0.171 (случайное угадывание с учётом баланса классов) до 0.399 в лучшем случае (LogReg).

2) Наибольший вклад в предсказание вероятности клика вносят:
    - app_id и site_id - идентификаторы рекламируемого приложения и веб-сайта показа. Это логично: конкретное приложение или сайт имеют кликабельность, которая переросла в хорошую связь при Таргет кодировании и получила высокий вес в моделях.
    - device_model - модель устройства пользователя. Тип устройства может влиять на поведение на сайте
    - ml_feature_9 и ml_feature_10 - машинно-сгенерированные признаки. Анонимны, но имеют и немалые веса, и коэффициенты корреляции с таргетом.

#### 11.3 Рекомендации

1. Создание признаков: к примеру site_category * device_type, app_id * hour. Это может добавить предсказательную силу для признаков, которые могли получить малый вес по отдельности
2. Из признака `hour` можно вычленить какие-либо временные метки и проанализировать связь с таргетом
3. Расширение методов калибровки (использование калибровки ПЛатта)

## 12. Сохранение модели для продакшена

#### 12.1 Сохранение артефактов

In [ ]:
# сохранение препроцессора и откалиброванной модели
joblib.dump(preprocessor, 'preprocessor.joblib')
joblib.dump(calib_svc, 'calibrated_model.joblib')

In [ ]:
# информация о признаках
feature_info = {
    "numeric_features": numeric_features,
    "binary_features": binary_features,
    "onehot_cat_features": onehot_cat_features,
    "target_cat_features": target_cat_features,
    "all_features": numeric_features + binary_features + onehot_cat_features + target_cat_features,
    "dropped_features": [
        "id",           # уникальный идентификатор — удалён в EDA
        "device_id",    # высокая кардинальность — удалён на этапе отбора признаков
        "device_ip",    # высокая кардинальность — удалён на этапе отбора признаков
        "ml_feature_1", # нулевая phi_k с таргетом
        "ml_feature_3", # нулевая phi_k с таргетом
        "ml_feature_4", # нулевая phi_k с таргетом (бинарный флаг)
        "banner_pos",   # высокая корреляция с другим признаком
        "C1",           # высокая корреляция с другим признаком
        "device_type",  # высокая корреляция с другим признаком
        "C17",          # высокая корреляция с другим признаком
        "site_domain",  # высокая корреляция с site_id
        "site_category",# высокая корреляция с site_id
        "app_domain",   # высокая корреляция с app_id
        "app_category", # высокая корреляция с app_id
    ],
    "encoding_strategy": {
        "numeric": "StandardScaler (после SimpleImputer median)",
        "binary": "без масштабирования (SimpleImputer most_frequent)",
        "low_cardinality_cat": "OneHotEncoder (кардинальность <= 10)",
        "high_cardinality_cat": "TargetEncoder (cv=5, smooth=30)"
    }
}
joblib.dump(feature_info, 'feature_info.joblib')

#### 12.2 Проверка работоспособности кода

In [ ]:
loaded_calib_svc = joblib.load('calibrated_model.joblib')
y_load = loaded_calib_svc.predict_proba(X_test)[:, 1]
load_pr_auc = average_precision_score(y_test, y_load)
init_pr_auc = average_precision_score(y_test, y_svc_score_calib)
print(f"PR-AUC загруженной модели: {round(load_pr_auc, 4)}")
print(f"PR-AUC исходной модели: {round(init_pr_auc, 4)}")

Метрика совпала. Сохраненные артефакты корректны.

---
Все материалы проекта доступны на GitHub:

https://github.com/DrichBMSTU/ad-click-ctr-prediction